### initialize

In [1]:
# Run this no matter which cells you want to run
import sys
sys.path.append(r'D:\Neural-Pipeline\source')

directory = r"D:\Neural-Pipeline\data"
# directory = r"\\172.30.3.33\homes\fetschlab\data\zarya\zarya_neuro"
subject = "zarya"
date = "20260417"



## 1. Merge recording file
> The Openephys file struct = "YYYY-MM-DD_HH-MM-SS\Record Node 101\experiment?\recording?" 

In [ ]:
from preprocessing1_openephys.DataProcessor import MergeRecordingFile

processor = MergeRecordingFile(directory, subject, date)
processor.extract_channel_positions(AP_name='ProbeA')
processor.extract_channel_positions(AP_name='ProbeB')
processor.merge_ap_data(num_channels=384, AP_name = 'ProbeB') #new ver:384; old ver:385
processor.merge_ap_data(num_channels=384, AP_name = 'ProbeA') #new ver:384; old ver:385
processor.merge_ttl_data()

# processor.merge_eye_data()
# processor.check_electrode_consistency()  ## check if the electrode configuration are consistent

## 2. Run Kilosort
> Don't forget to log the information in RecSectionInfo.ipynb

> In cmd: 
- conda activate kilosort
- python -m kilosort

## 3. Curation
> In cmd:
- conda activate phy2
- phy2 template-gui 'path-to-kilo\params.py'

> Track the good / mua units

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
import os

phy = True
phy_dir_name = os.path.join(directory, date, 'kilosort4_ProbeB_phy')
kilo_dir_name = os.path.join(directory, date, 'kilosort4_ProbeB')
phy_dir = Path(phy_dir_name) / 'cluster_info.tsv'
kilo_dir = Path(kilo_dir_name) / 'cluster_KSLabel.tsv'
cluster_group = pd.read_csv(phy_dir, sep='\t')
kilo_cluster = pd.read_csv(kilo_dir, sep='\t')


# Count KSLabel (Kilosort output)
kslabel_counts = kilo_cluster['KSLabel'].value_counts(dropna=False)

# Count group (Phy manual curation)
if phy:
    group_counts = cluster_group['group'].value_counts(dropna=False)

# Report both
print("=== Kilosort Labels (KSLabel) ===")
print(kslabel_counts)
if phy: 
    print("\n=== Phy Curated Labels (group) ===")
    print(group_counts)


## 4. Create event / unit struct (after kilosort)
> Create event / unit struct for .mat file

! need to work on saccade data

In [2]:
from preprocessing1_openephys.DataProcessor import CreateEventStruct
from preprocessing1_openephys.DataProcessor import CreateUnitStruct

event_processor = CreateEventStruct(directory, subject, date, AP_name='ProbeB')
event_processor.filter_events()
event_processor.load_info_data()
event_processor.process_trials()
event_processor.save_events_to_mat()
 
# kilo = "kilosort4_ProbeB_phy"
# # kilo = "kilosort4_phy"
# unit_processor = CreateUnitStruct(directory, subject, date, kilo, AP_name='ProbeB')
# unit_processor.build_unit_structure()
# unit_processor.save_units_to_mat()

Processing dots3DMP: 3 blocks
[128  15  15 ...   8   9   9]
[128  16   2 ...   7   7   8]
[128  17  17  35  35  35   2   3   3   4   5   5   5   5   5   5   6   6
   7   7   7   8   9   9   2   3   3   4   5   5   5   5   5   5   6   6
   7   7   7   8   9   9   2   3   3   4   5   5   5   5   5   5   6   6
   7   7   7   8   9   9   2   3   3   4   5   5   5   5   5   5   6   6
   7   7   7   8   9   9   2   3   3   4   5   5   5   5   5   5   6   6
   7   7   7   8   9   9   2   3   3   3   3   4   5   5   5   5   5   5
   6   6   7   7   7   8   9   9   2   3   3   4   5   5   5   5   5   5
   6   6   7   7   7   8   9   9   2   3   3   4   5   5   5   5   5   5
   6   6   7   7   7   8   9   9   2   3   3   4   5   5   5   5   5   5
   6   6   7   7   7   8   9   9   2   3   3   4   5   5   5   5  10  10
   2   3   3   4  10  10   2   3   3   4   5   5   5   5   5   5   6   6
   7   7   7   8   9   9   2   3   3   4   5   5   5   5  10  10   2   3
   3   4   5   5   5   5   5   5  

## 5. Create eyeXY struct
> no pupil data yet


In [ ]:
from preprocessing1_openephys.DataProcessor import CreateEyeXYStruct

unit_processor = CreateEyeXYStruct(directory, subject, date)
unit_processor.build_eyeXY_structure()
unit_processor.save_eyeXY_to_mat()